In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1).astype(np.float32)
cosine_similarity = np.clip(cosine_similarity, -1.0, 1.0).astype(np.float32)

# Original fixed linear map from cosine in [-1, 1] to STS score in [0, 5]
pred_linear_0_5 = (2.5 * (cosine_similarity + 1.0)).astype(np.float32)

# Alternative deterministic closed-form calibration:
# map cosine to angular similarity in [0, 1], then to [0, 5]
# score = 5 * (1 - arccos(cosine) / pi)
pred_angular_0_5 = (5.0 * (1.0 - (np.arccos(cosine_similarity) / np.pi))).astype(np.float32)

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["pred_linear_0_5"] = pred_linear_0_5
results_df["pred_angular_0_5"] = pred_angular_0_5
results_df["calibration_delta"] = (results_df["pred_angular_0_5"] - results_df["pred_linear_0_5"]).astype(np.float32)

results_df["absolute_error_linear"] = np.abs(results_df["pred_linear_0_5"] - results_df["label"]).astype(np.float32)
results_df["squared_error_linear"] = np.square(results_df["pred_linear_0_5"] - results_df["label"]).astype(np.float32)
results_df["signed_error_linear"] = (results_df["pred_linear_0_5"] - results_df["label"]).astype(np.float32)
results_df["agreement_gap_linear"] = np.abs(results_df["cosine_similarity"] - (results_df["label"] / 2.5 - 1.0)).astype(np.float32)

results_df["absolute_error_angular"] = np.abs(results_df["pred_angular_0_5"] - results_df["label"]).astype(np.float32)
results_df["squared_error_angular"] = np.square(results_df["pred_angular_0_5"] - results_df["label"]).astype(np.float32)
results_df["signed_error_angular"] = (results_df["pred_angular_0_5"] - results_df["label"]).astype(np.float32)
results_df["agreement_gap_angular"] = np.abs(np.cos(np.pi * (1.0 - results_df["label"] / 5.0)) - results_df["cosine_similarity"]).astype(np.float32)

print(results_df[[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "pred_linear_0_5", "pred_angular_0_5", "calibration_delta",
    "absolute_error_linear", "absolute_error_angular"
]].head(10))

In [ ]:
pearson_linear = pearsonr(results_df["pred_linear_0_5"], results_df["label"]).statistic
spearman_linear = spearmanr(results_df["pred_linear_0_5"], results_df["label"]).statistic
mae_linear = float(results_df["absolute_error_linear"].mean())
rmse_linear = float(np.sqrt(results_df["squared_error_linear"].mean()))

pearson_angular = pearsonr(results_df["pred_angular_0_5"], results_df["label"]).statistic
spearman_angular = spearmanr(results_df["pred_angular_0_5"], results_df["label"]).statistic
mae_angular = float(results_df["absolute_error_angular"].mean())
rmse_angular = float(np.sqrt(results_df["squared_error_angular"].mean()))

label_bin_edges = [-0.001, 1.0, 2.0, 3.0, 4.0, 5.001]
label_bin_names = ["[0,1)", "[1,2)", "[2,3)", "[3,4)", "[4,5]"]
results_df["label_bin"] = pd.cut(
    results_df["label"],
    bins=label_bin_edges,
    labels=label_bin_names,
    include_lowest=True,
    right=False,
)

agreement_gap_table = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        cosine_mean=("cosine_similarity", "mean"),
        linear_pred_mean=("pred_linear_0_5", "mean"),
        angular_pred_mean=("pred_angular_0_5", "mean"),
        agreement_gap_linear_mean=("agreement_gap_linear", "mean"),
        agreement_gap_angular_mean=("agreement_gap_angular", "mean"),
        absolute_error_linear_mean=("absolute_error_linear", "mean"),
        absolute_error_angular_mean=("absolute_error_angular", "mean"),
    )
    .reset_index()
)

calibration_delta_table = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        calibration_delta_mean=("calibration_delta", "mean"),
        calibration_delta_abs_mean=("calibration_delta", lambda x: float(np.mean(np.abs(x)))),
        calibration_delta_min=("calibration_delta", "min"),
        calibration_delta_max=("calibration_delta", "max"),
        signed_error_linear_mean=("signed_error_linear", "mean"),
        signed_error_angular_mean=("signed_error_angular", "mean"),
    )
    .reset_index()
)

metrics_summary = pd.DataFrame([
    {
        "mapping": "linear_0_5",
        "pearson_correlation": float(pearson_linear),
        "spearman_correlation": float(spearman_linear),
        "mae_0_5": mae_linear,
        "rmse_0_5": rmse_linear,
    },
    {
        "mapping": "angular_0_5",
        "pearson_correlation": float(pearson_angular),
        "spearman_correlation": float(spearman_angular),
        "mae_0_5": mae_angular,
        "rmse_0_5": rmse_angular,
    },
])

print(metrics_summary.round(6))
print("AGREEMENT_GAP_TABLE")
print(agreement_gap_table.round(6))
print("CALIBRATION_DELTA_TABLE")
print(calibration_delta_table.round(6))

In [ ]:
pd.set_option("display.max_colwidth", 160)

best_linear_examples = (
    results_df.sort_values(
        by=["absolute_error_linear", "agreement_gap_linear", "label"],
        ascending=[True, True, False],
    )
    [[
        "sentence1", "sentence2", "label", "cosine_similarity",
        "pred_linear_0_5", "pred_angular_0_5", "calibration_delta",
        "absolute_error_linear", "absolute_error_angular",
        "signed_error_linear", "signed_error_angular"
    ]]
    .head(10)
    .reset_index(drop=True)
)

worst_linear_examples = (
    results_df.sort_values(
        by=["absolute_error_linear", "agreement_gap_linear", "label"],
        ascending=[False, False, False],
    )
    [[
        "sentence1", "sentence2", "label", "cosine_similarity",
        "pred_linear_0_5", "pred_angular_0_5", "calibration_delta",
        "absolute_error_linear", "absolute_error_angular",
        "signed_error_linear", "signed_error_angular"
    ]]
    .head(10)
    .reset_index(drop=True)
)

largest_calibration_shift_examples = (
    results_df.assign(calibration_delta_abs=np.abs(results_df["calibration_delta"]))
    .sort_values(by=["calibration_delta_abs", "absolute_error_linear"], ascending=[False, False])
    [[
        "sentence1", "sentence2", "label", "cosine_similarity",
        "pred_linear_0_5", "pred_angular_0_5", "calibration_delta",
        "absolute_error_linear", "absolute_error_angular"
    ]]
    .head(10)
    .reset_index(drop=True)
)

print("BEST_LINEAR_AGREEMENT_EXAMPLES")
print(best_linear_examples)
print("\nWORST_LINEAR_AGREEMENT_EXAMPLES")
print(worst_linear_examples)
print("\nLARGEST_CALIBRATION_SHIFT_EXAMPLES")
print(largest_calibration_shift_examples)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "num_examples": int(len(df)),
    "linear_pearson_correlation": round(float(pearson_linear), 6),
    "linear_spearman_correlation": round(float(spearman_linear), 6),
    "linear_mae_0_5": round(mae_linear, 6),
    "linear_rmse_0_5": round(rmse_linear, 6),
    "angular_pearson_correlation": round(float(pearson_angular), 6),
    "angular_spearman_correlation": round(float(spearman_angular), 6),
    "angular_mae_0_5": round(mae_angular, 6),
    "angular_rmse_0_5": round(rmse_angular, 6),
    "mean_calibration_delta": round(float(results_df["calibration_delta"].mean()), 6),
    "mean_abs_calibration_delta": round(float(np.mean(np.abs(results_df["calibration_delta"]))), 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)